### A simple notebook showing how to use the causal prior

In [1]:
%load_ext autoreload
%autoreload 2

### Example 1: create a dataset that samples from the prior

Here, an observational dataset is created using some simple default config files

In [10]:
from priordata_processing.Datasets.MakePurelyObservationalDataset import MakePurelyObservationalDataset

from priors.causal_prior.ExampleConfigs.Basic_Configs import default_sampling_config as prior_config
from priordata_processing.Datasets.ExampleConfigs.BasicConfigs import default_dataset_config as dataset_config
from priordata_processing.Datasets.ExampleConfigs.BasicConfigs import default_preprocessing_config as preprocessing_config

from torch.utils.data import DataLoader


In [ ]:
SEED = 124
make_dataset = MakePurelyObservationalDataset(   # initialize factory for creating datasets
        scm_config=prior_config,
        preprocessing_config=preprocessing_config,
        dataset_config=dataset_config
)

dataset = make_dataset.create_dataset(seed=SEED)  #actually create dataset



c:\Users\arik_\miniconda3\envs\tabpfn-legacy2\lib\site-packages\xgboost\data.py:112: UserWarning: Use subset (sliced data) of np.ndarray is not recommended because it will generate extra copies and increase memory consumption
  warnings.warn(


In [ ]:
X_train, Y_train, X_test, Y_test = dataset[0]  # each element in the dataset is sampled on the fly and contains train and test splits

In [ ]:
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)   #you can directly use a the dataset above in a dataloader to sample batches. This is useful if you want to use multiprocessing for the sampling

sample_batch = next(iter(dataloader)) # each batch contains X_train, Y_train, X_test, Y_test with batch size 32

c:\Users\arik_\miniconda3\envs\tabpfn-legacy2\lib\site-packages\xgboost\data.py:112: UserWarning: Use subset (sliced data) of np.ndarray is not recommended because it will generate extra copies and increase memory consumption
  warnings.warn(


### Example 2: Directly sampling from the SCM

This shows how, under the hood, data is sampled from the SCM

In [17]:
N_SAMPLES = 256
seed = 128

In [ ]:
from priors.causal_prior.scm.InspectSCMSamples import InspectSCMSamples
from priors.causal_prior.scm.SCMHyperparameterSampler import SCMHyperparameterSampler
from priors.causal_prior.scm.SCMBuilder import SCMBuilder
from priors.causal_prior.ExampleConfigs.Basic_Configs import default_sampling_config 

In [ ]:
config = default_sampling_config   # a simple default config for the hyperparameters of the 

sampler = SCMHyperparameterSampler(config, seed=seed)  # initialize the sampler for the hyperparameters that samples according to the config

print(sampler.get_parameter_summary())


SCM Hyperparameter Sampling Configuration:

num_nodes: discrete_uniform(low=3, high=8)
graph_edge_prob: beta(alpha=2, beta=3)
graph_seed: discrete_uniform(low=0, high=100000000)
xgboost_prob: uniform(low=0.0, high=0.3)
mechanism_seed: discrete_uniform(low=0, high=100000000)
mlp_nonlins: Fixed = tabicl
mlp_num_hidden_layers: discrete_uniform(low=0, high=2)
mlp_hidden_dim: categorical(choices=[8, 16, 32, 64])
mlp_activation_mode: Fixed = pre
mlp_node_shape: Fixed = (1,)
xgb_num_hidden_layers: Fixed = 0
xgb_hidden_dim: categorical(choices=[0, 16, 32, 64])
xgb_activation_mode: categorical(choices=['pre', 'post'])
xgb_node_shape: Fixed = (1,)
xgb_n_training_samples: discrete_uniform(low=100, high=500)
xgb_add_noise: Fixed = False
random_additive_std: Fixed = True
exo_std_distribution: categorical(choices=['gamma', 'pareto'], probabilities=[1.0, 0.0])
endo_std_distribution: categorical(choices=['gamma', 'pareto'], probabilities=[1.0, 0.0])
exo_std_mean: lognormal(mean=0.0, std=0.5)
exo_std_s

In [ ]:
sampled_params = sampler.sample()   # sample a set of hyperparameters for the SCM
builder = SCMBuilder(**sampled_params)  # build the SCM according to the sampled hyperparameters
scm = builder.build()

# Sample both exogenous and endogenous noise before generating data
scm.sample_exogenous(num_samples=N_SAMPLES)   
scm.sample_endogenous_noise(num_samples=N_SAMPLES)

r = scm.propagate(num_samples=N_SAMPLES)  # this is just to demonstrate how one dataset can be sampled
# the output r will be a dictionary with keys being the variable names and values being the sampled data arrays
